<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/AdvancedAgent/ipynb/bonus_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
apt install -y zstd pciutils lshw
curl -fsSL https://ollama.com/install.sh | sh
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-ollama ollama pypdf

In [1]:
!nohup ollama serve &

nohup: appending output to 'nohup.out'


In [ ]:
!ollama pull embeddinggemma:300m

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv

os.makedirs(name="resources", exist_ok=True)
_ = load_dotenv(dotenv_path=".env", override=True)

## Semantic search

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")
data = loader.load()
print(data)

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'resources/acmecorp-employee-handbook.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Employee Handbook\nNon-Disclosure Agreement (NDA) Policy\nEmployees must protect confidential information belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprietary algorithms, financial information, and unreleased features. Confidential information may not\nbe shared with unauthorized individuals inside or outside the organization. These obligations continue\nafter employment ends.\nWorkplace Conduct Policy\nEmployees must maintain a respectful, professional environment

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(documents=data)

print(len(all_splits))

3


In [5]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="embeddinggemma:300m"
)

In [6]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embeddings)

In [7]:
ids = vector_store.add_documents(documents=all_splits)

In [8]:
results = vector_store.similarity_search(
    "How many days of vacation does an employee get in their first year?"
)

print(results[0])

page_content='prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per year (0.833 days per month)  1–3 years of service: 15 days per year (1.25 days per month)  3+
years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal
needs, or illness. Requests should be submitted in advance through the HR system unless related to
an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended
absences exceeding 5 consecutive business days require manager approval.
Travel & Expense Policy
Employees may be reimbursed for reasonable and necessary expenses incurred during approved
business travel. This includes transportation, lodging, meals, and incidental expenses within
established limits. Receipts must be submitted within 14 days of travel. First-class travel, personal' metadata={'producer': 'ReportL

In [9]:
import os
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gemma3:27b-cloud",
    model_provider="ollama",
    base_url="https://ollama.com",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

## RAG Agent

In [10]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information."
    )

In [12]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}
)

In [13]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='How many days of vacation does an employee get in their first year?', additional_kwargs={}, response_metadata={}, id='667f7159-07ad-48f7-9894-54ba238f0f73'),
              AIMessage(content='Please provide me with the employee handbook! I need the text of the handbook to be able to search for the vacation policy and tell you how many days an employee gets in their first year. \n\nJust paste the text here, or tell me how to access it (e.g., a link to a document).\n', additional_kwargs={}, response_metadata={'model': 'gemma3:27b-cloud', 'created_at': '2026-03-23T06:26:58.785854714Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2018210918, 'load_duration': None, 'prompt_eval_count': 38, 'prompt_eval_duration': None, 'eval_count': 66, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma3:27b-cloud', 'model_provider': 'ollama'}, id='lc_run--019d1960-0589-7863-9507-853af8d13a52-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'